# Continuity equation: Lagrangian advection and Eulerian flux

This notebook generates `fig:dynamic-continuity-equation`.  A smooth affine vector field advects a Gaussian law,
$$
    \dot x(t)=v(x(t)), \qquad \alpha_t=(T_t)_\sharp\alpha_0,
$$
and the same evolution satisfies the Eulerian continuity equation
$$
    \partial_t\alpha_t+\operatorname{div}(\alpha_t v)=0.
$$
The panels show trajectories, density contours at selected times, and the density flux $m_t=\rho_t v_t$ when $\alpha_t=\rho_t\,dx$.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")
for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from PIL import Image
import ot
from scipy.linalg import expm, solve
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter
from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY, DIRAC_MARKER_SIZE,
    setup_matplotlib, figure_dir, save_pdf, remove_axes, box_axes,
    interp_color, draw_point_clouds, draw_transport_segments, padded_limits,
)
setup_matplotlib()
rng = np.random.default_rng(2027)

In [2]:
NAME='dynamic-continuity-equation'; out=figure_dir(NAME)
A=np.array([[0.08,-0.72],[0.46,-0.14]]); b=np.array([0.65,0.12])
mu0=np.array([-0.72,-0.20]); S0=np.array([[0.16,0.055],[0.055,0.075]])
def M(t): return expm(t*A)
def q(t): return solve(A,(M(t)-np.eye(2))@b,assume_a='gen')
def push(P,t): return P@M(t).T+q(t)
def moments(t): return M(t)@mu0+q(t), M(t)@S0@M(t).T
def vel(P): return P@A.T+b
def density_grid(mean,cov,X,Y):
    G=np.stack([X,Y],axis=-1); inv=np.linalg.inv(cov); det=np.linalg.det(cov); D=G-mean
    return np.exp(-0.5*np.einsum('...i,ij,...j->...',D,inv,D))/(2*np.pi*np.sqrt(det))
pts=rng.multivariate_normal(mu0,S0,34); times=np.linspace(0,1,80); paths=np.stack([push(pts,t) for t in times],axis=1)
allp=paths.reshape(-1,2); xlim,ylim=padded_limits(allp,pad=0.10)
gx=np.linspace(xlim[0],xlim[1],90); gy=np.linspace(ylim[0],ylim[1],80); X,Y=np.meshgrid(gx,gy)
# Lagrangian panel.
fig,ax=plt.subplots(figsize=(4.55,1.85))
segs=[]; cols=[]
for path in paths[:26]:
    for k in range(len(times)-1):
        segs.append([path[k],path[k+1]]); cols.append((*interp_color(float(times[k])),0.34))
ax.add_collection(LineCollection(segs,colors=cols,linewidths=0.55,zorder=1))
for t,c in [(0,RED),(0.55,VIOLET),(1,BLUE)]:
    Z=push(pts,t); ax.scatter(Z[:,0],Z[:,1],s=DIRAC_MARKER_SIZE*0.72,color=c,marker='o',edgecolor='none',linewidth=0,zorder=3,alpha=0.88)
    mean,cov=moments(t); den=density_grid(mean,cov,X,Y); ax.contour(X,Y,den,levels=4,colors=[c],linewidths=0.58,alpha=0.55)
ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig,out/'advection.pdf',pad_inches=0.045); plt.close(fig)
# Flux panel.
t=0.55; mean,cov=moments(t); den=density_grid(mean,cov,X,Y); step=9; U=vel(np.stack([X[::step,::step],Y[::step,::step]],axis=-1).reshape(-1,2)).reshape(X[::step,::step].shape+(2,)); D=den[::step,::step]
fig,ax=plt.subplots(figsize=(2.45,1.85))
ax.contourf(X,Y,den,levels=10,colors=[interp_color(t)],alpha=0.13)
ax.contour(X,Y,den,levels=5,colors=[interp_color(t)],linewidths=0.55,alpha=0.70)
ax.quiver(X[::step,::step],Y[::step,::step],D*U[...,0],D*U[...,1],color=GRAY,angles='xy',scale_units='xy',scale=0.55,width=0.006,alpha=0.72)
ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig,out/'flux.pdf',pad_inches=0.045); plt.close(fig)